In [12]:
import polars as pl

# CSV 文件路径
churn_csv = '/Users/zealot/yizhou/git/data_research/com/silq/canclled_reason.csv'
stripe_csv = '/Users/zealot/yizhou/git/data_research/com/silq/stripe_subscription.csv'
df = pl.read_csv(churn_csv)
df2 = pl.read_csv(stripe_csv)
ctx = pl.SQLContext()
ctx.register("churn", df)
ctx.register("stripe", df2)


<SQLContext [tables:2] at 0x7fd56045f390>

## **1. Clean the data**
What issues did you find across both files and how did you handle them?

### 1.1 数据缺失：Days Since Last Lesson有很多null值，也许是查询错误，也有可能是无法正常计算，通常可以使用平均值或者是中位数代替，或者默认0。包括Name数据有缺失，但是并不影响数据分析。
### 1.2 年龄异常：这里有包括5，,6，,7岁的用户，与其他的分布相差过大，可能是用户随意填写的

### 2.1 What % of cancellations are trial abandonment vs paying customers churning?

In [17]:
ctx.execute("""
SELECT
    canceled_in_trial, count(*) cnt
FROM stripe where status!='active'
group by canceled_in_trial
""").collect()

canceled_in_trial,cnt
str,u32
"""No""",5
"""Yes""",11


#### Approximately 69% of cancellations occurred during the trial period, and 31% were from paying customers

### 2.2 We showed an offer to every churner and saved 0. Why and what would you change?

### There are multiple reasons:

1. User unfamiliarity: Users may not know how to use our product, leading to high learning costs. For example, login issues can be addressed through after-sales follow-up.

2. Seasonal factors: Over 50% of users are 25 or younger, likely students, who tend to use the product mainly during holidays, showing cyclical usage habits.

3. Account issues: Payment bugs, sound bugs, deduction errors, or login anomalies.

4. Insufficient product value: Users may not find courses corresponding to the language they wish to learn, or the quality of existing courses could be improved.

### Actionable Recommendation
It’s very likely that users who see the offer aren’t churning because of the price. We can also determine whether a user is price-sensitive through certain methods—for example, by tracking the number of visits or the time spent on the payment page.

In [27]:
df_grouped = ctx.execute("""
SELECT
    CASE
        WHEN "Lessons Completed" = 0 THEN '0 lessons'
        WHEN "Lessons Completed" BETWEEN 1 AND 5 THEN '1-5 lessons'
        WHEN "Lessons Completed" BETWEEN 6 AND 20 THEN '6-20 lessons'
        WHEN "Lessons Completed" > 20 THEN '20+ lessons'
        ELSE 'Unknown'
    END AS lesson_group,
    COUNT(*) AS churned_users
FROM churn
""").collect()

### 2.3 Break down churners by lessons completed: 0, 1–5, 6–20, 20+. What does it tell you?

In [36]:
ctx.execute("""
SELECT
    CASE
        WHEN "Lessons Completed" = 0 THEN '0 lessons'
        WHEN "Lessons Completed" BETWEEN 1 AND 5 THEN '1-5 lessons'
        WHEN "Lessons Completed" BETWEEN 6 AND 20 THEN '6-20 lessons'
        WHEN "Lessons Completed" > 20 THEN '20+ lessons'
        ELSE 'Unknown'
    END AS lesson_group,
    COUNT(*) AS churned_users
FROM churn
GROUP BY
    CASE
        WHEN "Lessons Completed" = 0 THEN '0 lessons'
        WHEN "Lessons Completed" BETWEEN 1 AND 5 THEN '1-5 lessons'
        WHEN "Lessons Completed" BETWEEN 6 AND 20 THEN '6-20 lessons'
        WHEN "Lessons Completed" > 20 THEN '20+ lessons'
        ELSE 'Unknown'
    END
ORDER BY churned_users DESC
""").collect()

lesson_group,churned_users
list[str],u32
"[""1-5 lessons"", ""1-5 lessons"", … ""1-5 lessons""]",29
"[""0 lessons"", ""0 lessons"", … ""0 lessons""]",27
"[""6-20 lessons"", ""6-20 lessons"", … ""6-20 lessons""]",21
"[""20+ lessons"", ""20+ lessons"", … ""20+ lessons""]",7


### Of all churned users:

32% completed 0 lessons,

35% completed 1–5 lessons,

25% completed 6–20 lessons,

8% completed 20+ lessons.

### Issue Identified: Activation Problem

Users who churned after 0 or 1–5 lessons account for 66% of total churned users (27 + 29), indicating that most users leave after completing only a few lessons. The core problem is not long-term retention but the initial user experience and early-stage activation.

### Actionable Recommendation:

Product optimization should prioritize the initial activation and the experience of the first 5 lessons.

### 2.4 Which churn profile represents the most recoverable revenue? Show your math.

To estimate recoverable revenue and prioritize retention:

Low-value churn: Users who churned during the trial period or are Basic members, generally with low lesson completion. These users have limited engagement and little revenue contribution.

High-value churn: Users who have already converted to paid Premium subscriptions and completed a meaningful number of lessons. These users represent the highest recoverable revenue and are worth targeted retention efforts, such as discounts, personalized content, or engagement campaigns.

This classification combines payment tier (Premium/Basic), trial completion, and lesson engagement, giving a more actionable view of which users to prioritize for retention.

### 2.5 42% said "not sure" if they improved. Cross-reference that with their lesson count. What do you find?
In the provided sample dataset, 50% of churned users selected "not sure", although the assessment prompt references 42%.


In [41]:
total_not_sure = ctx.execute("""
SELECT COUNT(*) AS total_not_sure
FROM churn
WHERE "Progress Perception" = 'not_sure'
""").collect()

total_not_sure_value = total_not_sure["total_not_sure"][0]
total_not_sure_value

not_sure_by_lessons = ctx.execute("""
SELECT
    CASE
        WHEN "Lessons Completed" = 0 THEN '0 lessons'
        WHEN "Lessons Completed" BETWEEN 1 AND 5 THEN '1-5 lessons'
        WHEN "Lessons Completed" BETWEEN 6 AND 20 THEN '6-20 lessons'
        WHEN "Lessons Completed" > 20 THEN '20+ lessons'
        ELSE 'Unknown'
    END AS lesson_group,
    COUNT(*) AS users
FROM churn
WHERE "Progress Perception" = 'not_sure'
GROUP BY
    CASE
        WHEN "Lessons Completed" = 0 THEN '0 lessons'
        WHEN "Lessons Completed" BETWEEN 1 AND 5 THEN '1-5 lessons'
        WHEN "Lessons Completed" BETWEEN 6 AND 20 THEN '6-20 lessons'
        WHEN "Lessons Completed" > 20 THEN '20+ lessons'
        ELSE 'Unknown'
    END
ORDER BY users DESC
""").collect()

not_sure_by_lessons = not_sure_by_lessons.with_columns(
    (pl.col("users") * 100 / total_not_sure_value)
    .round(1)
    .alias("share_pct")
)

not_sure_by_lessons

lesson_group,users,share_pct
list[str],u32,f64
"[""0 lessons"", ""0 lessons"", … ""0 lessons""]",20,47.6
"[""1-5 lessons"", ""1-5 lessons"", … ""1-5 lessons""]",14,33.3
"[""6-20 lessons"", ""6-20 lessons"", … ""6-20 lessons""]",7,16.7
"[""20+ lessons""]",1,2.4


In [42]:
early_not_sure = ctx.execute("""
SELECT COUNT(*) AS users_5_or_less
FROM churn
WHERE "Progress Perception" = 'not_sure'
  AND "Lessons Completed" <= 5
""").collect()

early_not_sure = early_not_sure.with_columns(
    (pl.col("users_5_or_less") * 100 / total_not_sure_value)
    .round(1)
    .alias("share_pct")
)

early_not_sure

users_5_or_less,share_pct
u32,f64
34,81.0


### Key Finding

81% of users who were unsure whether they had improved completed five or fewer lessons.

### Actionable Recommendation

The product should focus on helping users experience an early sense of progress within the first five lessons.

Potential improvements include:

Progress milestones
Vocabulary mastery indicators
Completion achievements
Personalized feedback
Clear learning goals

The objective is to make progress visible before users decide to cancel.

## 3. Design a dashboard
### What's the one dashboard you'd build in week 1? What does it show and what decision does it drive?
